In [ ]:
import os
import sys

from pyspark.sql import SparkSession

# Use the Python executable from the active environment
python_path = sys.executable

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

# Prefer local loopback for Spark networking
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

spark = (
    SparkSession.builder
    .appName("PySparkDemo")
    .master("local[2]")
    .config("spark.pyspark.python", python_path)
    .config("spark.pyspark.driver.python", python_path)
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)
print("SparkSession created successfully.")

In [ ]:
data = [
    (101, "Amit Kumar Sharma", "IT", 75000, "Mumbai", "98765-43210"),
    (102, "Neha Patel", "HR", 55000, "Pune", "98765-12345"),
    (103, "Rahul Verma", "IT", 80000, "Delhi", "98765-67890"),
    (104, "Priya Singh", "Finance", 65000, "Mumbai", "98765-54321"),
    (105, "Ravi Kumar", "IT", 75000, "Pune", "98765-98765"),
    (106, "Sneha Shah", "HR", 60000, "Delhi", "98765-11111")
]

columns = [
    "EmpID",
    "FullName",
    "Department",
    "Salary",
    "City",
    "Phone"
]

df = spark.createDataFrame(data, columns)

df.show(truncate=False)

In [ ]:
df.select(
    "EmpID",
    "FullName",
    "Salary"
).show(truncate=False)

In [ ]:
from pyspark.sql.functions import col

df.select(
    col("FullName"),
    col("Department"),
    col("Salary")
).show()

In [ ]:
df.select(
    col("FullName"),
    col("Salary"),
    (col("Salary") * 12).alias("AnnualSalary")
).show()

In [ ]:
df.filter(
    col("Salary") > 65000
).show()

In [ ]:
df.select(
    col("FullName"),
    col("Department"),
    col("Salary"),
    (col("Salary") * 12).alias("AnnualSalary")
).filter(
    col("Salary") > 65000
).show()

In [ ]:
df.filter(
    col("Department") == "IT"
).show()

In [ ]:
df.filter(
    (col("Department") == "IT") &
    (col("Salary") > 75000)
).show()

In [ ]:
df.filter(
    (col("City") == "Mumbai") |
    (col("City") == "Delhi")
).show()

In [ ]:
df.filter(
    col("Department").isin("IT", "HR")
).show()

In [ ]:
df.filter(
    "Salary >= 60000 AND Department = 'IT'"
).show()

In [ ]:
df.select(
    col("EmpID").alias("EmployeeID"),
    col("FullName").alias("EmployeeName"),
    col("Salary").alias("MonthlySalary")
).show()

In [ ]:
df.select(
    col("FullName"),
    (col("Salary") * 0.10).alias("Bonus")
).show()

In [ ]:
employees = df.alias("e")

employees.select(
    col("e.EmpID"),
    col("e.FullName")
).show()

In [ ]:
df.show()

In [ ]:
df.select("Department").distinct().show()
df.select("City").distinct().show()

In [ ]:
duplicate_data = [
    (101, "Amit", "IT"),
    (102, "Neha", "HR"),
    (101, "Amit", "IT"),
    (103, "Rahul", "Finance"),
    (102, "Neha", "HR")
]

duplicate_df = spark.createDataFrame(
    duplicate_data,
    ["EmpID", "Name", "Department"]
)

duplicate_df.show()

In [ ]:
unique_df = duplicate_df.distinct()

unique_df.show()

In [ ]:
duplicate_df.dropDuplicates().show()

In [ ]:
duplicate_df.dropDuplicates(
    ["EmpID"]
).show()

In [ ]:
duplicate_df.dropDuplicates(
    ["Name", "Department"]
).show()

In [ ]:
salary_data = [
    (101, "Amit", 50000),
    (101, "Amit", 60000),
    (102, "Neha", 55000),
    (103, "Rahul", 70000)
]

salary_df = spark.createDataFrame(
    salary_data,
    ["EmpID", "Name", "Salary"]
)

salary_df.dropDuplicates(["EmpID"]).show()

In [ ]:
bonus_df = df.withColumn(
    "Bonus",
    col("Salary") * 0.10
)

bonus_df.show()

In [ ]:
annual_df = df.withColumn(
    "AnnualSalary",
    col("Salary") * 12
)

annual_df.select(
    "FullName",
    "Salary",
    "AnnualSalary"
).show()

In [ ]:
from pyspark.sql.functions import lit

company_df = df.withColumn(
    "Company",
    lit("ABC Technologies")
)

company_df.show()

In [ ]:
updated_salary_df = df.withColumn(
    "Salary",
    col("Salary") * 1.10
)

updated_salary_df.select(
    "FullName",
    "Salary"
).show()

In [ ]:
result_df = df \
    .withColumn(
        "AnnualSalary",
        col("Salary") * 12
    ) \
    .withColumn(
        "Bonus",
        col("Salary") * 0.10
    ) \
    .withColumn(
        "Company",
        lit("ABC Technologies")
    )

result_df.show()

In [ ]:
renamed_df = df.withColumnRenamed(
    "FullName",
    "EmployeeName"
)

renamed_df.show()

In [ ]:
renamed_df = df \
    .withColumnRenamed(
        "EmpID",
        "EmployeeID"
    ) \
    .withColumnRenamed(
        "FullName",
        "EmployeeName"
    ) \
    .withColumnRenamed(
        "Salary",
        "MonthlySalary"
    )

renamed_df.printSchema()

In [ ]:
from pyspark.sql.functions import when

salary_category_df = df.withColumn(
    "SalaryCategory",
    when(
        col("Salary") >= 75000,
        "High"
    ).otherwise("Normal")
)

salary_category_df.select(
    "FullName",
    "Salary",
    "SalaryCategory"
).show()

In [ ]:
salary_grade_df = df.withColumn(
    "SalaryGrade",
    when(
        col("Salary") >= 75000,
        "Grade A"
    ).when(
        col("Salary") >= 60000,
        "Grade B"
    ).otherwise("Grade C")
)

salary_grade_df.select(
    "FullName",
    "Salary",
    "SalaryGrade"
).show()

In [ ]:
bonus_df = df.withColumn(
    "Bonus",
    when(
        col("Department") == "IT",
        col("Salary") * 0.15
    ).when(
        col("Department") == "HR",
        col("Salary") * 0.10
    ).otherwise(
        col("Salary") * 0.05
    )
)

bonus_df.select(
    "FullName",
    "Department",
    "Salary",
    "Bonus"
).show()

In [ ]:
null_data = [
    (101, "Amit", 75000),
    (102, "Neha", None),
    (103, "Rahul", 80000)
]

null_df = spark.createDataFrame(
    null_data,
    ["EmpID", "Name", "Salary"]
)

result_df = null_df.withColumn(
    "Salary",
    when(
        col("Salary").isNull(),
        lit(0)
    ).otherwise(
        col("Salary")
    )
)

result_df.show()

In [ ]:
from pyspark.sql.functions import regexp_replace

phone_df = df.withColumn(
    "CleanPhone",
    regexp_replace(
        col("Phone"),
        "-",
        ""
    )
)

phone_df.select(
    "FullName",
    "Phone",
    "CleanPhone"
).show()

In [ ]:
phone_df = df.withColumn(
    "CleanPhone",
    regexp_replace(
        col("Phone"),
        "[^0-9]",
        ""
    )
)

phone_df.select(
    "Phone",
    "CleanPhone"
).show()

In [ ]:
names_data = [
    (1, "Amit@Sharma"),
    (2, "Neha#Patel"),
    (3, "Rahul$Verma")
]

names_df = spark.createDataFrame(
    names_data,
    ["ID", "Name"]
)

clean_names_df = names_df.withColumn(
    "CleanName",
    regexp_replace(
        col("Name"),
        "[^a-zA-Z]",
        " "
    )
)

clean_names_df.show()

In [ ]:
spaces_data = [
    (1, "Amit  Kumar  Sharma"),
    (2, "    Neha     Patel   "),
    (3, "Rahul  Verma")
]

spaces_df = spark.createDataFrame(
    spaces_data,
    ["ID", "Name"]
)

clean_spaces_df = spaces_df.withColumn(
    "CleanName",
    regexp_replace(
        col("Name"),
        r"\s+",
        " "
    )
)

clean_spaces_df.show()

In [ ]:
from pyspark.sql.functions import split

split_df = df.withColumn(
    "NameParts",
    split(
        col("FullName"),
        " "
    )
)

split_df.select(
    "FullName",
    "NameParts"
).show(truncate=False)

In [ ]:
name_df = df \
    .withColumn(
        "FirstName",
        split(col("FullName"), " ")[0]
    ) \
    .withColumn(
        "LastName",
        split(col("FullName"), " ")[1]
    )

name_df.select(
    "FullName",
    "FirstName",
    "LastName"
).show()

In [ ]:
email_data = [
    (101, "amit@gmail.com"),
    (102, "neha@yahoo.com"),
    (103, "rahul@outlook.com")
]

email_df = spark.createDataFrame(
    email_data,
    ["EmpID", "Email"]
)

result_df = email_df \
    .withColumn(
        "Username",
        split(col("Email"), "@")[0]
    ) \
    .withColumn(
        "Domain",
        split(col("Email"), "@")[1]
    )

result_df.show(truncate=False)

In [ ]:
skills_data = [
    (101, "Python,SQL,Spark"),
    (102, "Java,Spring,SQL"),
    (103, "Python,Azure,Databricks")
]

skills_df = spark.createDataFrame(
    skills_data,
    ["EmpID", "Skills"]
)

result_df = skills_df.withColumn(
    "SkillsList",
    split(col("Skills"), ",")
)

result_df.show(truncate=False)